 ##### Step 1: First I am loading all the source tables

In [4]:
from pyspark.sql.functions import *

df_customers = spark.read.table("bronze_olist_customers_dataset")
df_order_items = spark.read.table("bronze_olist_order_items_dataset")
df_order_payments = spark.read.table("bronze_olist_order_payments_dataset")
df_orders = spark.read.table("bronze_olist_orders_dataset")
df_products = spark.read.table("bronze_olist_products_dataset")


StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 6, Finished, Available, Finished, False)

In [ ]:
# Checking if all tables are loaded in dataframe correctly
# display(df_customers)
# display(df_order_items)
# display(df_order_payments)
# display(df_orders)
# display(df_products)

##### Step 2: Handling customers without customer_id (Basically null customer_id)

In [10]:
df_customers_clean = df_customers.filter("customer_id is not null")

#Note: We can use any other way also if business users need customer 
#records without id also for that we can then predecide and keep some id for such users

StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 12, Finished, Available, Finished, False)

##### Step 3: Casting columns as per requirement for order_items

In [18]:
from pyspark.sql.functions import col 
df_order_items_clean = df_order_items.withColumn('shipping_limit_date',col('shipping_limit_date').cast('date'))\
.withColumn('price',col('price').cast('float'))\
.withColumn('freight_value',col('freight_value').cast('float'))

StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 20, Finished, Available, Finished, False)

##### Step 4: Handling negative and null payment values and casting as per need

In [37]:
from pyspark.sql.functions import col

df_order_payments_clean = df_order_payments.filter(
    (col("payment_value") >= 0) & (col("payment_value").isNotNull())
).withColumn('payment_installments',col('payment_installments').cast('int'))\
.withColumn('payment_value',col('payment_value').cast('float'))

StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 39, Finished, Available, Finished, False)

##### Step 5: Changing date columns for orders as per required format

In [60]:
from pyspark.sql.functions import to_timestamp
df_orders_clean = df_orders\
.withColumn('order_purchase_timestamp',to_timestamp(col('order_purchase_timestamp'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_approved_at',to_timestamp(col('order_approved_at'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_delivered_carrier_date',to_timestamp(col('order_delivered_carrier_date'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_delivered_customer_date',to_timestamp(col('order_delivered_customer_date'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_estimated_delivery_date',to_timestamp(col('order_estimated_delivery_date'),'yyyy-MM-dd HH:mm:ss'))

StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 62, Finished, Available, Finished, False)

##### Step 6: For products casting of numeric columns is done

In [64]:
df_products_clean = df_products\
.withColumn('product_name_lenght',col('product_name_lenght').cast('int'))\
.withColumn('product_description_lenght',col('product_description_lenght').cast('int'))\
.withColumn('product_photos_qty',col('product_photos_qty').cast('int'))\
.withColumn('product_weight_g',col('product_weight_g').cast('float'))\
.withColumn('product_length_cm',col('product_length_cm').cast('float'))\
.withColumn('product_height_cm',col('product_height_cm').cast('float'))\
.withColumn('product_width_cm',col('product_width_cm').cast('float'))


StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 66, Finished, Available, Finished, False)

##### Step 7: Writing clean dataframes to Silver tables

In [66]:
df_customers_clean.write.mode("overwrite").saveAsTable("silver_customers")
df_order_items_clean.write.mode("overwrite").saveAsTable("silver_order_items")
df_order_payments_clean.write.mode("overwrite").saveAsTable("silver_order_payments")
df_orders_clean.write.mode("overwrite").saveAsTable("silver_orders")
df_products_clean.write.mode("overwrite").saveAsTable("silver_products")


StatementMeta(, a7a86237-4a80-42ca-9a3e-0dd42f8ac10c, 68, Finished, Available, Finished, False)